In [16]:
import os
import numpy as np
import pandas as pd
import cv2
import librosa
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from moviepy.editor import VideoFileClip



# Load ResNet model for feature extraction
def create_visual_model():
    """Create a visual feature extraction model using ResNet50."""
    base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
    input_layer = layers.Input(shape=(224, 224, 3))
    x = base_model(input_layer)
    return models.Model(inputs=input_layer, outputs=x)

def create_audio_model(input_shape):
    """Create an audio feature extraction model."""
    input_layer = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, kernel_size=3, activation='relu')(input_layer)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Conv1D(128, kernel_size=3, activation='relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Flatten()(x)
    return models.Model(inputs=input_layer, outputs=x)




def extract_visual_features(frames, model):
    """Extract visual features from video frames using ResNet model."""
    features = []
    for frame in frames:
        # Preprocess the frame for ResNet
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (224, 224))
        frame = np.expand_dims(frame, axis=0) / 255.0  # Normalize
        feature = model.predict(frame)
        features.append(feature)
    return np.array(features)

# Function to extract audio from a video
def extract_audio(video_path):
    clip = VideoFileClip(video_path)
    audio = clip.audio
    audio_path = video_path.replace(".mp4", ".wav")
    audio.write_audiofile(audio_path)
    return audio_path


def extract_audio_features(audio_path):
    """Extract MFCC features from audio."""
    y, sr = librosa.load(audio_path, sr=None)
    if y is None or sr is None:  # Check if audio data is loaded
        print(f"Warning: Audio data not loaded for {audio_path}.")
        return np.zeros(13)  # Return zero array if loading fails
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return np.mean(mfccs.T, axis=0)  # Average over time

# dynamic_sampling

def sample_frames(video_path, max_frames = 10):
    
    '''
    A function for frame extraction
    
    args:
        video_path (str): path to the video file
        max_frames (int): maximum number of frames to extract   
        frame_size (int; tuple): size of the frames to extract 
        
    return:
        frames (list): list of extracted frames
        '''
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_interval = max(1, total_frames // max_frames)
    
    frames = []
    frame_count = 0
    sampled_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_count % sample_interval == 0:
            frame = cv2.resize(frame, max_frames)  # Resizing the frames based on the pretrained model used
            frames.append(frame)
            sampled_count += 1
        
        frame_count += 1
    
    cap.release()
    # Padding with black frames (0s) if the total frames is less than max_frames.
    while len(frames) < max_frames:
        frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
    
    return np.array(frames[:max_frames])

def preprocess_video_data(video_dir, metadata,visual_model):
    """Preprocess video data to extract audio and visual features."""
    video_features_list, audio_features_list = [], []
    labels = []
    
    for index, row in metadata.iterrows():
        video_id = row['VideoID']
        video_path = os.path.join(video_dir, f'{video_id}.mp4')
        
        # Sample frames from the video
        video_frames = sample_frames(video_path, n_frames=10)
        
        # Check if any frames were extracted
        if len(video_frames) == 0:
            print(f"Warning: No features extracted for video {video_id}. Skipping this video.")
            continue
        
        # Extract visual features using ResNet50
        video_features = extract_visual_features(video_frames, visual_model)
        video_features_list.append(video_features)
        
        # Extract audio features
        audio_path = extract_audio(video_path)  # Ensure this function extracts audio correctly
        audio_features = extract_audio_features(audio_path)
        audio_features_list.append(audio_features)
        
        # Append the label (how funny)
        labels.append(row['how_funny_encoded'])
    
    return np.array(video_features_list), np.array(audio_features_list), np.array(labels)

def train_and_evaluate_model(video_features, audio_features, labels):
    """Train and evaluate a deep learning model using intermediate fusion."""
    # Flatten video features to combine with audio features
    video_features_flat = video_features.reshape(video_features.shape[0], -1)
    
    # Combine video and audio features
    combined_features = np.concatenate((video_features_flat, audio_features), axis=1)

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.2, random_state=42)

    # Build the intermediate fusion model
    video_input = layers.Input(shape=(video_features_flat.shape[1],))
    audio_input = layers.Input(shape=(audio_features.shape[1],))

    # Define video branch
    video_branch = layers.Dense(128, activation='relu')(video_input)
    video_branch = layers.Dense(64, activation='relu')(video_branch)

    # Define audio branch
    audio_branch = layers.Dense(128, activation='relu')(audio_input)
    audio_branch = layers.Dense(64, activation='relu')(audio_branch)

    # Fusion layer
    merged = layers.concatenate([video_branch, audio_branch])
    merged = layers.Dense(128, activation='relu')(merged)
    output = layers.Dense(4, activation='softmax')(merged)  # Assuming 4 labels for 'how funny'

    # Create and compile model
    model = models.Model(inputs=[video_input, audio_input], outputs=output)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Train the model
    model.fit([X_train, X_train], y_train, epochs=10, batch_size=32, validation_split=0.2)

    # Evaluate the model
    y_pred = model.predict([X_test, X_test])
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred_classes)
    f1 = f1_score(y_test, y_pred_classes, average='weighted')
    conf_matrix = confusion_matrix(y_test, y_pred_classes)
    
    print(f'Accuracy: {accuracy:.2f}')
    print(f'F1 Score: {f1:.2f}')
    print(f'Confusion Matrix:\n{conf_matrix}')
    
    return model

In [17]:
# Example usage
# Ensure that the video directory and metadata file path are correctly set

video_dir = "/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/videos"
metadata_file = "/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/excel.xlsx"
metadata = pd.read_excel(metadata_file)

# Initialize models
visual_model = create_visual_model()
audio_model = create_audio_model((13, None))  # Adjust audio input shape if necessary

# Preprocess video data and metadata
video_features, audio_features, labels = preprocess_video_data(video_dir, metadata, visual_model)

# Train the model
model = train_and_evaluate_model(video_features, audio_features, labels)


TypeError: unsupported operand type(s) for %: 'NoneType' and 'int'

# Delecting Files

In [15]:
import os

def delete_wav_files(directory_path):
    """
    Deletes all .wav audio files in the specified directory.

    Parameters:
    directory_path (str): The path to the directory where .wav files will be deleted.
    """
    try:
        # List all files in the specified directory
        for filename in os.listdir(directory_path):
            # Check if the file ends with .wav
            if filename.lower().endswith('.wav'):
                file_path = os.path.join(directory_path, filename)  # Create full file path
                os.remove(file_path)  # Delete the file
                print(f"Deleted: {file_path}")  # Print confirmation of deletion

        print("Deletion process completed.")
        
    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage
#delete_wav_files('/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/videos')

In [ ]:
#23 

In [1]:
import pandas as pd
import numpy as np
import cv2
import librosa
import os
from moviepy.editor import VideoFileClip
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Step 1: Data Ingestion
def load_metadata(metadata_path):
    """Load metadata from an Excel file."""
    try:
        metadata = pd.read_excel(metadata_path)
        print(f"Metadata loaded successfully from {metadata_path}.")
        return metadata
    except Exception as e:
        print(f"Error loading metadata: {e}")
        raise

def extract_audio_from_video(video_path):
    """Extract audio from a video file."""
    if not os.path.exists(video_path):
        print(f"Error: Video file not found: {video_path}")
        return None
    try:
        video = VideoFileClip(video_path)
        audio = video.audio
        return audio.to_soundarray(fps=22000)  # Extract audio as an array
    except Exception as e:
        print(f"Error processing video for {video_path}: {e}")
        return None

def extract_visual_features(video_path):
    """Extract visual features from a video file."""
    if not os.path.exists(video_path):
        print(f"Error: Video file not found: {video_path}")
        return None
    
    visual_features = []
    cap = cv2.VideoCapture(video_path)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # Resize frame and convert to grayscale for feature extraction
        frame = cv2.resize(frame, (64, 64))  # Resize to 64x64
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        visual_features.append(gray_frame.flatten())  # Flatten the frame to a 1D array

    cap.release()
    return np.array(visual_features)

def extract_movement_features(video_path):
    """Extract movement features from a video file."""
    if not os.path.exists(video_path):
        print(f"Error: Video file not found: {video_path}")
        return None
    
    movement_features = []
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_frame = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        # Calculate the absolute difference between the current frame and the previous frame
        movement = cv2.absdiff(prev_frame, gray_frame)
        movement_features.append(movement.flatten())  # Flatten the movement array
        prev_frame = gray_frame

    cap.release()
    return np.array(movement_features)

# Step 2: Feature Extraction
def align_features(metadata, video_dir):
    """Align features and labels from metadata."""
    audio_features_list = []
    visual_features_list = []
    movement_features_list = []
    labels = []
    
    for index, row in metadata.iterrows():
        video_filename = row['VideoID']  # Assuming this column exists
        video_path = os.path.join(video_dir, video_filename)

        # Check if the video file exists
        if not os.path.isfile(video_path):
            print(f"Error: Could not open video file {video_path}.")
            continue  # Skip to the next file if it doesn't exist

        # Extract audio features
        audio_features = extract_audio_from_video(video_path)
        if audio_features is None:
            continue  # Skip if audio extraction failed

        # Extract visual features
        visual_features = extract_visual_features(video_path)
        if visual_features is None:
            continue  # Skip if visual extraction failed

        # Extract movement features
        movement_features = extract_movement_features(video_path)
        if movement_features is None:
            continue  # Skip if movement extraction failed

        # Assuming labels are in a column named 'label'
        label = row['HowFunny']
        audio_features_list.append(audio_features)
        visual_features_list.append(visual_features)
        movement_features_list.append(movement_features)
        labels.append(label)

    return (np.array(audio_features_list), 
            np.array(visual_features_list), 
            np.array(movement_features_list), 
            np.array(labels))

# Step 3: Model Training
def train_model(audio_features, visual_features, movement_features, labels):
    """Train a Random Forest model."""
    # Combine features into a single array for training
    combined_features = np.concatenate((audio_features, visual_features, movement_features), axis=1)
    X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.2, random_state=42)
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    return model, X_test, y_test

# Step 4: Evaluation
def evaluate_model(model, X_test, y_test):
    """Evaluate the trained model."""
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

# Main pipeline
def main(metadata_path, video_dir):
    metadata = load_metadata(metadata_path)
    audio_features, visual_features, movement_features, labels = align_features(metadata, video_dir)
    
    if audio_features.size == 0 or labels.size == 0:
        print("No features or labels to train the model.")
        return
    
    model, X_test, y_test = train_model(audio_features, visual_features, movement_features, labels)
    evaluate_model(model, X_test, y_test)

# Example usage
if __name__ == "__main__":
    main('../data/excel.xlsx', '../data/videos/')

Metadata loaded successfully from ../data/excel.xlsx.
Error: Could not open video file ../data/videos/2UWdXP.joke3.rep1.take1.ThatsNotAHat.mp4.


: 